#  GlobalClinic AI — Early Disease Detection with Gemma 4


## Project Summary
GlobalClinic AI is an offline-capable, multilingual clinical decision support system targeting rural clinics in low-resource settings. It fine-tunes Gemma 4 (E4B)  on tuberculosis, pneumonia, malaria, and malnutrition datasets, then wraps the model in a RAG pipeline grounded in WHO/CDC guidelines.

### Architecture
```
[Patient symptoms / X-ray]
        ↓
[Gemma 4 E4B — LoRA fine-tuned on TBX11K + NIH ChestX-ray14 + MedQA]
        ↓
[RAG — FAISS vector store of WHO/CDC guidelines (offline)]
        ↓
[Structured triage output → FHIR DiagnosticReport]
        ↓
[ClinIQ Web UI — 6 languages, works on Android tablet / Raspberry Pi]
```

### Models Used
- `google/gemma-4-e4b-it` — base model (edge-optimised)
- `google/medgemma-4b-it` — medical imaging specialist

### Datasets
- TBX11K (11,200 TB chest X-rays)
- NIH ChestX-ray14 (112,120 images, 14 labels)
- MedQA USMLE-4 (clinical reasoning QA)
- PubMedQA (biomedical literature QA)

---

## 1. Environment Setup

In [1]:
# Install all required packages
!pip install -U "unsloth[colab-new]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 751.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
import subprocess

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
Tesla T4, 15360 MiB, 14910 MiB
Tesla T4, 15360 MiB, 14910 MiB



In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Pull HuggingFace token from Kaggle secrets
# Add your token at: Kaggle → Account → Settings → Secrets → HF_TOKEN
try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('✅ HuggingFace login successful')
except Exception as e:
    print(f'⚠️  HF token not found in secrets: {e}')
    print('   You can still run inference; fine-tuned weights won\'t push to Hub')

✅ HuggingFace login successful


## 2. Configuration

In [4]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ClinIQConfig:
    # ── Model ──────────────────────────────────────────────────────────────────
    model_id: str           = 'google/gemma-4-e4b-it'
    # Swap to 'google/medgemma-4b-it' for imaging-focused fine-tune

    # ── Quantisation (keeps E4B within T4 16 GB VRAM) ─────────────────────────
    load_in_4bit: bool      = True

    # ── LoRA ───────────────────────────────────────────────────────────────────
    lora_r: int             = 16
    lora_alpha: int         = 32
    lora_dropout: float     = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ])

    # ── Training ───────────────────────────────────────────────────────────────
    output_dir: str         = '/kaggle/working/cliniq-gemma4'
    num_epochs: int         = 3
    batch_size: int         = 2
    grad_accum: int         = 8     # effective batch = 16
    learning_rate: float    = 2e-4
    max_seq_length: int     = 2048
    bf16: bool              = True

    # ── Data ───────────────────────────────────────────────────────────────────
    max_train_samples: int  = 2000  # increase for full run

    # ── HuggingFace Hub ────────────────────────────────────────────────────────
    push_to_hub: bool       = False  # set True after testing
    hub_model_id: str       = 'your-username/cliniq-gemma4-e4b-medical'
    # Naming follows Gemma variant guidelines:
    # {base}-{size}-{task/domain}-{optional-descriptor}
    # → cliniq-gemma4-e4b-medical

CFG = ClinIQConfig()
print(f'Model      : {CFG.model_id}')
print(f'Output dir : {CFG.output_dir}')
print(f'LoRA rank  : {CFG.lora_r}')
print(f'Push to Hub: {CFG.push_to_hub}')

Model      : google/gemma-4-e4b-it
Output dir : /kaggle/working/cliniq-gemma4
LoRA rank  : 16
Push to Hub: False


## 3. Dataset Loading & Prompt Formatting

In [5]:
SYSTEM_PROMPT = """You are ClinIQ, an AI medical assistant for healthcare workers
in resource-limited rural clinics. You assist with:
- Preliminary triage and screening (TB, pneumonia, malaria, malnutrition)
- Symptom analysis and differential diagnosis
- Interpreting chest X-rays and clinical findings
- Generating structured referral recommendations

Always state your confidence level (High / Medium / Low).
Always recommend specialist follow-up for serious findings.
You do NOT replace clinical judgment."""


def format_symptom_prompt(sample: dict) -> str:
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n"
        f"Patient: {sample.get('age','?')}y {sample.get('sex','?')}.\n"
        f"Complaint: {sample.get('chief_complaint','')}\n"
        f"Symptoms: {sample.get('symptoms','')}\n"
        f"Duration: {sample.get('duration','?')}\n"
        f"Vitals: {sample.get('vitals','Not recorded')}\n"
        f"Setting: Rural clinic, limited diagnostics\n"
        f"Provide: assessment, differential, recommended action.<end_of_turn>\n"
        f"<start_of_turn>model\n{sample.get('answer','')}\n<end_of_turn>"
    )


def format_medqa_prompt(sample: dict) -> str:
    opts = '\n'.join(f"{k}. {v}" for k,v in sample.get('options',{}).items())
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n{sample.get('question','')}\n\n{opts}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"Answer: **{sample.get('answer_idx','')}** — {sample.get('answer','')}\n"
        f"Reasoning: {sample.get('explanation','Based on clinical guidelines.')}<end_of_turn>"
    )


def format_xray_prompt(sample: dict) -> str:
    return (
        f"<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"<start_of_turn>user\n"
        f"Interpret this chest X-ray. Provide:\n"
        f"1. Key radiological findings\n2. Most likely diagnosis (confidence %)\n"
        f"3. Differentials\n4. Next steps for a rural clinic\n[IMAGE]<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"**Findings:** {sample.get('findings','')}\n"
        f"**Diagnosis:** {sample.get('label','')}\n"
        f"**Action:** Initiate appropriate isolation and referral if infectious aetiology suspected.<end_of_turn>"
    )


print(' Prompt templates defined')

 Prompt templates defined


In [6]:
from datasets import load_dataset, Dataset
import json, random

all_samples = []

# ── 1. MedQA USMLE ─────────────────────────────────────────────────────────────
print('Loading MedQA USMLE…')
try:
    ds_medqa = load_dataset('GBaker/MedQA-USMLE-4-options', split='train')
    for row in ds_medqa.select(range(min(800, len(ds_medqa)))):
        all_samples.append({
            'text': format_medqa_prompt({
                'question':   row['question'],
                'options':    row['options'],
                'answer_idx': row['answer_idx'],
                'answer':     row['options'].get(row['answer_idx'],''),
            }),
            'source': 'medqa'
        })
    print(f'  MedQA: {min(800,len(ds_medqa))} samples')
except Exception as e:
    print(f'  MedQA: {e}')

# ── 2. PubMedQA ────────────────────────────────────────────────────────────────
print('Loading PubMedQA…')
try:
    ds_pubmed = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
    for row in ds_pubmed.select(range(min(400, len(ds_pubmed)))):
        ctx = ' '.join(row.get('context',{}).get('contexts',[]))[:400]
        all_samples.append({
            'text': format_symptom_prompt({
                'chief_complaint': row.get('question',''),
                'symptoms': ctx,
                'answer':   row.get('long_answer', row.get('final_decision',''))
            }),
            'source': 'pubmedqa'
        })
    print(f'  PubMedQA: {min(400,len(ds_pubmed))} samples')
except Exception as e:
    print(f'  PubMedQA: {e}')

# ── 3. NIH ChestX-ray14 ────────────────────────────────────────────────────────
print('Loading NIH ChestX-ray14…')
try:
    ds_nih = load_dataset('alkzar90/NIH-Chest-X-ray-dataset', split='train',
                           trust_remote_code=True)
    disease_map = {
        'Pneumonia':'bacterial or viral pneumonia',
        'Tuberculosis':'active tuberculosis',
        'Effusion':'pleural effusion',
        'Infiltration':'pulmonary infiltration',
        'No Finding':'no significant abnormality'
    }
    for row in ds_nih.select(range(min(600, len(ds_nih)))):
        label_raw = row.get('Finding Labels','No Finding').split('|')[0]
        label = disease_map.get(label_raw, label_raw.lower())
        all_samples.append({
            'text': format_xray_prompt({
                'label':    label,
                'findings': f'Radiograph demonstrating {label}'
            }),
            'source': 'nih_cxr'
        })
    print(f'  NIH CXR: {min(600,len(ds_nih))} samples')
except Exception as e:
    print(f'   NIH CXR: {e}')

# ── 4. TBX11K (if uploaded to Kaggle dataset) ──────────────────────────────────
import os
tbx_path = '/kaggle/input/tbx11k/annotations.json'
if os.path.exists(tbx_path):
    print('Loading TBX11K…')
    with open(tbx_path) as f:
        anns = json.load(f)
    for ann in anns.get('images',[]):
        label = 'active TB' if ann.get('category')=='tb' else 'normal'
        all_samples.append({
            'text': format_xray_prompt({'label':label,'findings':ann.get('findings',f'Pattern consistent with {label}')}),
            'source':'tbx11k'
        })
    print(f' TBX11K: {len([x for x in all_samples if x["source"]=="tbx11k"])} samples')
else:
    print('   TBX11K not attached — add it as a Kaggle dataset input')

# ── Shuffle & cap ──────────────────────────────────────────────────────────────
random.shuffle(all_samples)
all_samples = all_samples[:CFG.max_train_samples]

hf_ds = Dataset.from_list(all_samples)
split  = hf_ds.train_test_split(test_size=0.1, seed=42)

print(f'\n📊 Dataset ready: {len(split["train"])} train | {len(split["test"])} val')

Loading MedQA USMLE…


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'alkzar90/NIH-Chest-X-ray-dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  PubMedQA: 400 samples
Loading NIH ChestX-ray14…


README.md: 0.00B [00:00, ?B/s]

NIH-Chest-X-ray-dataset.py: 0.00B [00:00, ?B/s]

   NIH CXR: Dataset scripts are no longer supported, but found NIH-Chest-X-ray-dataset.py
   TBX11K not attached — add it as a Kaggle dataset input

📊 Dataset ready: 1080 train | 120 val


## 4. Load Gemma 4 with QLoRA

In [7]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

In [8]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",   
    max_seq_length=2048,
    dtype=None,                           
    load_in_4bit=True,                    
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,                         # Unsloth-optimised: keep at 0
    bias="none",
    use_gradient_checkpointing="unsloth",   # 30% extra VRAM savings
)
model.print_trainable_parameters()
print("Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


trainable params: 42,401,792 || all params: 8,038,558,240 || trainable%: 0.5275
Model loaded


## 5. Fine-Tuning

In [9]:
from trl import SFTConfig
import torch

sft_config = SFTConfig(
    output_dir="./outputs",

    # MEMORY SAVERS
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # IMPORTANT
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    # optimizer
    learning_rate=2e-4,
    num_train_epochs=3,

    # huge VRAM saver
    gradient_checkpointing=True,

    logging_steps=10,

    save_strategy="epoch",
    eval_strategy="no",   # disable eval first
)

In [10]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import torch, os

class CFG:
    output_dir      = "./outputs"
    num_epochs      = 3
    batch_size      = 2
    grad_accum      = 4
    learning_rate   = 2e-4     # also fixed: 2e-5 is too low for LoRA
    max_seq_length  = 1024
    eval_save_steps = 100      # keep eval and save in sync

os.makedirs(CFG.output_dir, exist_ok=True)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    # Paths
    output_dir                  = CFG.output_dir, 
    # SFT-specific — MUST live in SFTConfig
    dataset_text_field          = "text",
    packing                     = True,
  # Training
    num_train_epochs            = CFG.num_epochs,
    per_device_train_batch_size = CFG.batch_size,
    per_device_eval_batch_size  = CFG.batch_size,
    gradient_accumulation_steps = CFG.grad_accum,
    learning_rate               = CFG.learning_rate,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,

    # Precision
    bf16                        = use_bf16,
    fp16                        = not use_bf16,

    # Logging
    logging_steps               = 10,
    report_to                   = "none",
  # Eval + save — steps MUST match for EarlyStoppingCallback
    eval_strategy               = "steps",
    eval_steps                  = CFG.eval_save_steps,
    save_strategy               = "steps",
    save_steps                  = CFG.eval_save_steps,   # same as eval_steps
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
)

# ── SFTTrainer only gets model, tokenizer, datasets, args, callbacks ───────────
trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    args           = training_args,
    train_dataset  = split["train"],
    eval_dataset   = split["test"],
    callbacks      = [EarlyStoppingCallback(early_stopping_patience=3)],
)
print(f"Precision  : {'bf16' if use_bf16 else 'fp16'}")
print(f"Trainable  : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Training started…")
trainer.train()
print("Training complete")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1080 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/120 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,080 | Num Epochs = 3 | Total steps = 405
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 42,401,792 of 8,038,558,240 (0.53% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.328724,3.317949
200,0.293606,3.357337
300,0.251949,3.419771
400,0.239600,3.390528


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-400/tokenizer_config.json.


Training complete


## 6. Save & Push to HuggingFace Hub

In [11]:
import os

CFG.output_dir = "/kaggle/working/fine_tuned_model"
save_path = f"{CFG.output_dir}/final"

os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

# List saved files
for f in os.listdir(save_path):
    size = os.path.getsize(f"{save_path}/{f}") / 1e6
    print(f"{f:40s} {size:.1f} MB")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/fine_tuned_model/final/tokenizer_config.json.


Model saved to: /kaggle/working/fine_tuned_model/final
README.md                                0.0 MB
processor_config.json                    0.0 MB
tokenizer_config.json                    0.0 MB
chat_template.jinja                      0.0 MB
adapter_model.safetensors                169.7 MB
adapter_config.json                      0.0 MB
training_args.bin                        0.0 MB
tokenizer.json                           32.2 MB


## 8. RAG Pipeline — Build Offline Index

In [12]:
pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
import json
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
from pathlib import Path

# Built-in WHO knowledge chunks (same as rag_pipeline.py BUILTIN_KNOWLEDGE)
KNOWLEDGE = [
    {'content': 'TB diagnosis: Consider in patients with cough ≥2 weeks, weight loss, night sweats, hemoptysis. Use GeneXpert MTB/RIF where available. Treatment: 2HRZE/4HR. Notify public health and initiate contact tracing.', 'source': 'WHO TB Guidelines 2022', 'keywords': ['tuberculosis','TB','cough','GeneXpert','sputum']},
    {'content': 'Community pneumonia: CRB-65 scoring. First-line: Amoxicillin 1g TID × 5 days. Refer if SpO₂ <92%, RR >30, confusion, inability to maintain orals.', 'source': 'WHO Pneumonia Guidelines', 'keywords': ['pneumonia','CRB-65','amoxicillin','SpO₂']},
    {'content': 'Malaria: Use RDT before treatment. Uncomplicated P. falciparum: Artemether-lumefantrine 6 doses over 3 days. Severe malaria: IV artesunate + urgent referral.', 'source': 'WHO Malaria Treatment 2023', 'keywords': ['malaria','RDT','artemether','artesunate','Plasmodium']},
    {'content': 'SAM (MUAC <11.5 cm): Inpatient stabilisation with F-75 therapeutic milk. No RUTF until oedema resolves and appetite returns. Danger signs: lethargy, convulsions, respiratory distress.', 'source': 'WHO/UNICEF CMAM Guidelines', 'keywords': ['malnutrition','MUAC','SAM','RUTF','F-75']},
    {'content': 'Hypertension: Grade 2 HTN ≥140/90. First-line: ACE inhibitor or CCB or thiazide. Refer urgently if BP >180/120 with end-organ damage.', 'source': 'WHO Hypertension Guidelines', 'keywords': ['hypertension','blood pressure','ACE','amlodipine']},
]

INDEX_DIR = Path('/kaggle/working/cliniq_index')
INDEX_DIR.mkdir(exist_ok=True)

print('Building offline FAISS index…')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
texts = [k['content'] for k in KNOWLEDGE]
embs  = embedder.encode(texts, normalize_embeddings=True).astype('float32')

index = faiss.IndexFlatIP(embs.shape[1])
index.add(embs)
faiss.write_index(index, str(INDEX_DIR / 'medical.faiss'))
with open(INDEX_DIR / 'documents.json', 'w') as f:
    json.dump(KNOWLEDGE, f)

print(f'Index built: {index.ntotal} vectors → {INDEX_DIR}')

Building offline FAISS index…


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built: 5 vectors → /kaggle/working/cliniq_index


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer

# -------------------------
# Load embedder
# -------------------------
embedder = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device='cpu'
)
def rag_retrieve(query: str, top_k: int = 3):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0: continue
        doc = KNOWLEDGE[idx].copy()
        doc['score'] = float(score)
        results.append(doc)
    return results

# Test retrieval
queries = [
    'patient with 3-week cough and night sweats',
    'child with low MUAC and bilateral edema',
    'fever headache positive malaria test',
]
for q in queries:
    docs = rag_retrieve(q, top_k=1)
    print(f'Q: {q[:55]}')
    print(f'   → {docs[0]["source"]} (score: {docs[0]["score"]:.3f})')
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q: patient with 3-week cough and night sweats
   → WHO TB Guidelines 2022 (score: 0.507)

Q: child with low MUAC and bilateral edema
   → WHO/UNICEF CMAM Guidelines (score: 0.333)

Q: fever headache positive malaria test
   → WHO Malaria Treatment 2023 (score: 0.521)

